# Timestral conversion check

Converts the four released Timestral checkpoints with the script from the transformers pull request and compares each converted model with the original implementation, with float32 weights and again with float64 weights.

If the two implementations compute the same thing, their difference is rounding error: it should shrink by orders of magnitude when the weights are float64.

Run all cells (Runtime > Run all). It takes a few minutes: most of it is installing the branch and downloading the weights.

In [ ]:
!pip install -q "git+https://github.com/Abdelaziz03/transformers@add-timestral" "git+https://github.com/mistralai/timestral.git@a534bdbd244d64f3d9d743f46c7bf6fc330ed4ff"

In [ ]:
import json

import pandas as pd
import torch
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from timestral import TimestralConfig as OriginalConfig
from timestral import TimestralForecaster
from timestral import TimestralModel as OriginalModel

from transformers.models.timestral.convert_timestral_original_to_hf import convert_checkpoint


def make_inputs(patch_length, block_size):
    # The same inputs as `verify_conversion` in the conversion script.
    generator = torch.Generator().manual_seed(0)
    context_length = 8 * patch_length + 5
    prediction_length = (block_size + 1) * patch_length + 3
    steps = torch.arange(context_length, dtype=torch.float32)
    targets = 10 * torch.sin(2 * torch.pi * steps / 24) + torch.randn(2, 2, context_length, generator=generator)
    targets[0, 0, 5:9] = float("nan")
    past = torch.randn(2, 1, context_length, generator=generator)
    known = torch.randn(2, 1, context_length + prediction_length, generator=generator)
    cases = {"targets only": {}, "with covariates": {"past_covariates": past, "known_covariates": known}}
    return targets, prediction_length, cases


def load_original(repo_id):
    with open(hf_hub_download(repo_id, "config.json"), encoding="utf-8") as config_file:
        config = json.load(config_file)
    return config, load_file(hf_hub_download(repo_id, "model.safetensors"))


@torch.no_grad()
def compare(name, config, state_dict):
    rows = []
    for dtype in (torch.float32, torch.float64):
        original = OriginalModel(OriginalConfig.from_dict(config))
        original.load_state_dict(state_dict, strict=True)
        forecaster = TimestralForecaster(original.eval().to(dtype))
        model = convert_checkpoint(config, state_dict).to(dtype)
        targets, prediction_length, cases = make_inputs(model.config.patch_length, model.config.block_size)
        for case, covariates in cases.items():
            expected = forecaster.forecast(targets, prediction_length=prediction_length, **covariates).quantiles.double()
            got = model(past_values=targets, prediction_length=prediction_length, **covariates).full_predictions.double()
            difference = (got - expected).abs()
            row = {
                "model": name,
                "weights": str(dtype).split(".")[-1],
                "case": case,
                "max |original|": expected.abs().max().item(),
                "max abs diff": difference.max().item(),
                "max diff / max |original|": difference.max().item() / expected.abs().max().item(),
            }
            for tolerance in (1e-4, 1e-3, 1e-2):
                failing = difference > tolerance + tolerance * expected.abs()
                row[f"elements failing rtol=atol={tolerance:g}"] = int(failing.sum())
            row["elements"] = difference.numel()
            rows.append(row)
    return rows

In [ ]:
rows = []
for size in ("5M", "28M", "123M", "209M"):
    config, state_dict = load_original(f"mistralai/Timestral-{size}")
    rows += compare(f"Timestral-{size}", config, state_dict)
    print(f"Timestral-{size} done")

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 20)
results = pd.DataFrame(rows)
print(results.to_string(index=False, float_format=lambda value: f"{value:.3g}"))

In [ ]:
# Values for the integration test of the pull request: Timestral-209M on a sine wave.
import math

config, state_dict = load_original("mistralai/Timestral-209M")
model = convert_checkpoint(config, state_dict)
steps = torch.arange(512, dtype=torch.float32)
past_values = torch.sin(2 * math.pi * steps / 24)[None, None, :]
with torch.no_grad():
    outputs = model(past_values=past_values, prediction_length=64)

torch.set_printoptions(precision=6, linewidth=200)
print("shape", tuple(outputs.full_predictions.shape))
print("median[:8]", outputs.mean_predictions[0, 0, :8])
print("median[-4:]", outputs.mean_predictions[0, 0, -4:])
print("q10[:4]", outputs.full_predictions[0, 0, :4, 2])
print("q90[:4]", outputs.full_predictions[0, 0, :4, 18])